# Metrics and reward transform

`env.metrics` is a `Metrics` that accumulates statistics automatically as `step()` runs. It is kept separate from the per-step output because it is evaluation data — not model input.

## What `metrics` accumulates

Whenever the wrapped env reports `terminated` or `truncated`, `step()` records that segment in `metrics`. Those flags mean whatever the wrapped env uses them for. If `EnvConfig.reward_transform` is set, the recorded return uses the transformed rewards (the same values written to the step output), including the reset-frame reward (`step_index == 0`).

**On a `SingleEnv`** (one env):

| Property | Type | Contents |
|----------|------|----------|
| `env.metrics.episode_cum_rewards` | `tuple[float, ...]` | Cumulative rewards for every completed segment |
| `env.metrics.episode_lengths` | `tuple[float, ...]` | Step counts for every completed segment |

**On a `GroupEnv`** (multiple envs), `env.metrics` is a `GroupMetrics` — a live read-through view with no own storage. Each property is a tuple with one entry per env.


## Imports

In [1]:
from mouse_gym import EnvConfig, make_env, make_group_env


## Single-env rollout

CartPole gives a reward of `+1` every step until the pole falls. Metrics accumulate those per-step rewards into returns automatically.


In [2]:
env = make_env(EnvConfig(id="CartPole-v1", seed=0, seed_mode="constant"))


## Rollout

Run 1000 random-action steps. `metrics` fills silently as episodes complete — no special handling is needed in the loop.


In [3]:
for _ in range(1000):
    env.step(env.action_space.sample())


## Inspect metrics results

After the rollout, read `env.metrics` to summarise all completed episodes. On a `SingleEnv`, `metrics` holds tuples — no per-env indexing needed.


In [4]:
rewards = env.metrics.episode_cum_rewards
lengths = env.metrics.episode_lengths

print(f"Episodes completed : {len(rewards)}")
print(f"Avg return         : {sum(rewards) / len(rewards):.1f}")
print(f"Avg episode length : {sum(lengths) / len(lengths):.1f} steps")
print()
print("First 5 returns     :", [f"{r:.1f}" for r in rewards[:5]])
print("First 5 lengths     :", lengths[:5])


Episodes completed : 50
Avg return         : 18.8
Avg episode length : 18.8 steps

First 5 returns     : ['13.0', '15.0', '23.0', '14.0', '11.0']
First 5 lengths     : (13.0, 15.0, 23.0, 14.0, 11.0)


## clear() between evaluation phases

Call `env.metrics.clear()` before the next evaluation window. The next read is an empty tuple.


In [5]:
env.metrics.clear()
print("After clear():")
print(f"  episode_cum_rewards = {env.metrics.episode_cum_rewards}")
print(f"  episode_lengths     = {env.metrics.episode_lengths}")

for _ in range(500):
    env.step(env.action_space.sample())

rewards2 = env.metrics.episode_cum_rewards
print(f"\nSecond window — episodes completed: {len(rewards2)}")
print(f"Second window — avg return    : {sum(rewards2) / len(rewards2):.1f}")

env.close()


After clear():
  episode_cum_rewards = ()
  episode_lengths     = ()

Second window — episodes completed: 20
Second window — avg return    : 24.2


## Multi-env aggregation

With multiple env instances, each inner list is independent. The example below runs CartPole (2 instances) and MountainCar (2 instances) together, then summarises per-instance and across all instances.


In [6]:
env = make_group_env([
    EnvConfig(id="CartPole-v1", seed=0, seed_mode="constant", name="CartPole-0"),
    EnvConfig(id="CartPole-v1", seed=1, seed_mode="constant", name="CartPole-1"),
    EnvConfig(id="MountainCar-v0", seed=2, seed_mode="constant", name="MountainCar-0"),
    EnvConfig(id="MountainCar-v0", seed=3, seed_mode="constant", name="MountainCar-1"),
])

for _ in range(500):
    env.step(env.action_space.sample())

print("Per-instance summary:")
for i, name in enumerate(env.names):
    ep_rewards = env.metrics.episode_cum_rewards[i]
    ep_lengths = env.metrics.episode_lengths[i]
    avg_r = sum(ep_rewards) / len(ep_rewards) if ep_rewards else float("nan")
    avg_l = sum(ep_lengths) / len(ep_lengths) if ep_lengths else float("nan")
    print(f"  {name:15s}  episodes={len(ep_rewards):3d}  avg_return={avg_r:7.1f}  avg_length={avg_l:.1f}")

all_returns = [r for env_rewards in env.metrics.episode_cum_rewards for r in env_rewards]
print(f"\nAll instances combined — total episodes: {len(all_returns)}")

env.close()


Per-instance summary:
  CartPole-0       episodes= 22  avg_return=   21.7  avg_length=21.7
  CartPole-1       episodes= 23  avg_return=   20.7  avg_length=20.7
  MountainCar-0    episodes=  2  avg_return= -200.0  avg_length=200.0
  MountainCar-1    episodes=  2  avg_return= -200.0  avg_length=200.0

All instances combined — total episodes: 49


## `reward_transform`

`reward_transform` rewrites each step reward, including reset frames, before the step output and before metrics. It is called with `step_index`, `state`, `action`, `reward`, `terminated`, `truncated`, and `next_state`. Accept unused fields with `**kwargs`.

A reset frame is `step_index == 0`. On that frame `reward` is `0.0`, `action` is `None`, and both flags are false. Env steps pass the raw Gymnasium reward.

The env below ends on the first real step, so every boundary is visible in a short loop. The transform charges `-0.1` on each reset frame and scales env rewards by `10`. Each recorded return is `-0.1 + 10.0`.


In [7]:
import gymnasium as gym


class OneStepEnv(gym.Env):
    """Reset obs `0`; one step returns obs `1`, reward `1.0`, terminated."""

    observation_space = gym.spaces.Discrete(2)
    action_space = gym.spaces.Discrete(2)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        return 0, {}

    def step(self, action):
        return 1, 1.0, True, False, {}


def reward_transform(*, step_index, reward, **_kwargs):
    if step_index == 0:
        return -0.1
    return reward * 10.0


env = make_env(
    EnvConfig(seed=0, seed_mode="constant", env_fn=OneStepEnv, reward_transform=reward_transform)
)
try:
    for _ in range(4):
        _observation, reward, terminated, truncated, _info = env.step(env.action_space.sample())
        print(
            f"step={env.step_index} reward={reward:.1f} "
            f"terminated={terminated} truncated={truncated}"
        )
    print("returns:", env.metrics.episode_cum_rewards)
    print("lengths:", env.metrics.episode_lengths)
finally:
    env.close()


step=0 reward=-0.1 terminated=False truncated=False
step=1 reward=10.0 terminated=True truncated=False
step=0 reward=-0.1 terminated=False truncated=False
step=1 reward=10.0 terminated=True truncated=False
returns: [9.9, 9.9]
lengths: [1.0, 1.0]
